# EnCodon BiDirectional: codon design by masked prediction

CodonFM/**Encodon** is a *bidirectional* (masked) codon language model. This notebook runs its core inference workflow:

- **Input:** an amino-acid sequence
  - seed it with an initial choice of synonymous codons (back-translation)
  - mask a subset of the codon tokens
- **Output:** predict the masked codon tokens from the surrounding bidirectional context

The prediction fills the masked positions with model-plausible codons, which we then inspect (how many stayed synonymous, and how the sequence's CodonFM fitness changed). It exercises the `codonfm-fitness` and `codonfm-sample` tools.

> **Requirements:** a CUDA GPU, and gated checkpoint access — set `HF_TOKEN` and accept the NVIDIA Open Model License on each `nvidia/NV-CodonFM-Encodon-*-v1` HuggingFace repo. Use `DEVICE = "cpu"` to run on CPU (slow).

In [ ]:
DEVICE = "cuda"
CHECKPOINT = "encodon_80m"

## Step 1 — input amino-acid sequence

The design target is a protein. Here we use a 24-residue fragment of GFP.

In [ ]:
PROTEIN = "MVSKGEELFTGVVPILVELDGDVN"  # 24 amino acids
print(f"input protein ({len(PROTEIN)} aa): {PROTEIN}")

## Step 2 — seed with initial synonymous codons

Back-translate the protein into a coding sequence by assigning one codon per residue. We reuse Proto Language's `STANDARD_GENETIC_CODE` (from the CAI constraint) and pick a fixed representative codon per amino acid — any valid synonymous choice works as the starting point the model will refine.

In [ ]:
from proto_language.constraint.sequence_composition.codon_adaptation_index_constraint import (
    STANDARD_GENETIC_CODE,
)

# amino acid -> its synonymous codons; pick a deterministic representative to seed with.
_aa_to_codons: dict[str, list[str]] = {}
for _codon, _aa in STANDARD_GENETIC_CODE.items():
    if _aa != "*":
        _aa_to_codons.setdefault(_aa, []).append(_codon)
PREFERRED_CODON = {aa: sorted(codons)[0] for aa, codons in _aa_to_codons.items()}


def back_translate(protein: str) -> str:
    """Seed a coding sequence from a protein by choosing one synonymous codon per residue."""
    return "".join(PREFERRED_CODON[aa] for aa in protein)


def translate(cds: str) -> str:
    """Translate a coding sequence back to its amino-acid sequence."""
    return "".join(STANDARD_GENETIC_CODE[cds[i : i + 3]] for i in range(0, len(cds), 3))


seed_cds = back_translate(PROTEIN)
assert translate(seed_cds) == PROTEIN  # the seed encodes exactly the input protein
print(f"seed CDS ({len(seed_cds)} nt / {len(seed_cds) // 3} codons):")
print(seed_cds)

## Step 3 — mask a subset of codons and predict them

`codonfm-sample` masks a subset of codon positions (here `num_mutations` of them) and predicts each from the bidirectional context in a single forward pass — the EnCodon masked-prediction step. Sequence length (and reading frame) is preserved.

In [ ]:
from proto_tools import CodonFMSampleConfig, CodonFMSampleInput, run_codonfm_sample

N_MASKED = 8  # number of codon tokens to mask and predict

predicted = run_codonfm_sample(
    CodonFMSampleInput(sequences=[seed_cds]),
    CodonFMSampleConfig(model_checkpoint=CHECKPOINT, num_mutations=N_MASKED, temperature=1.0, device=DEVICE, seed=0),
).sequences[0]

seed_codons = [seed_cds[i : i + 3] for i in range(0, len(seed_cds), 3)]
pred_codons = [predicted[i : i + 3] for i in range(0, len(predicted), 3)]
changed = [(k, s, p) for k, (s, p) in enumerate(zip(seed_codons, pred_codons)) if s != p]

print(f"predicted CDS: {predicted}")
print(f"\nthe model changed {len(changed)} codon(s):")
for k, s, p in changed:
    print(f"  codon {k:>2}: {s} ({STANDARD_GENETIC_CODE[s]}) -> {p} ({STANDARD_GENETIC_CODE[p]})")

## Step 4 — inspect the prediction

Two questions about the predicted tokens: how many kept the same amino acid (a *synonymous* codon swap), and did the sequence become more model-typical (higher CodonFM fitness)?

In [ ]:
from proto_tools import CodonFMFitnessConfig, CodonFMFitnessInput, run_codonfm_fitness

pred_protein = translate(predicted)
synonymous = sum(a == b for a, b in zip(PROTEIN, pred_protein))
print(f"protein identity: {synonymous}/{len(PROTEIN)} residues unchanged (synonymous predictions)")
if pred_protein != PROTEIN:
    diffs = [f"{a}{i}{b}" for i, (a, b) in enumerate(zip(PROTEIN, pred_protein)) if a != b]
    print(f"non-synonymous changes: {', '.join(diffs)}")

fit = run_codonfm_fitness(
    CodonFMFitnessInput(sequences=[seed_cds, predicted]),
    CodonFMFitnessConfig(model_checkpoint=CHECKPOINT, device=DEVICE),
)
seed_fit, pred_fit = (r.fitness for r in fit.results)
print(f"\nCodonFM fitness (mean codon log-likelihood): seed {seed_fit:.4f} -> predicted {pred_fit:.4f} ({pred_fit - seed_fit:+.4f})")

## (Optional) iterative bidirectional refinement

Repeating *mask a subset → predict* for several rounds lets the bidirectional model keep refining the codon choices. We track fitness each round. (The base model predicts codons freely, so the encoded protein can drift; constrain to synonymous codons if you need to preserve the exact protein.)

In [ ]:
def codonfm_fitness(seq: str) -> float:
    return run_codonfm_fitness(
        CodonFMFitnessInput(sequences=[seq]),
        CodonFMFitnessConfig(model_checkpoint=CHECKPOINT, device=DEVICE),
    ).results[0].fitness


current = seed_cds
history = [codonfm_fitness(current)]
for round_idx in range(5):
    current = run_codonfm_sample(
        CodonFMSampleInput(sequences=[current]),
        CodonFMSampleConfig(model_checkpoint=CHECKPOINT, num_mutations=N_MASKED, temperature=1.0, device=DEVICE, seed=round_idx),
    ).sequences[0]
    history.append(codonfm_fitness(current))

print("fitness by round:", [round(f, 4) for f in history])
print(f"final protein identity: {sum(a == b for a, b in zip(PROTEIN, translate(current)))}/{len(PROTEIN)} residues")
print(f"final CDS: {current}")